Copyright: Vrije Universiteit Amsterdam, Faculty of Humanities, CLTL

# Lab 1.6: Splitting the Dataset

When we run a neural language model, we are currently usually relying on a [Huggingface checkpoint](https://huggingface.co/models). To faciliate the processing, Huggingface has published a [code database](https://huggingface.co/docs/datasets/) for datasets. In this lab, we transform our dataset into the Huggingface format and split it into a train and a test portion.

## 1. Huggingface dataset

In [3]:
import pandas as pd

# We first check if the dataset is well formatted. If you get warnings, check the erroneous lines.
# We are currently ignoring the wrongly formatted lines
pandas_dataset = pd.read_csv(
    "../results/mediastack_results/veganism_overview.tsv", sep="\t", on_bad_lines="warn"
)

C:\Users\NIKHIL\AppData\Local\Temp\ipykernel_14792\2114363821.py:5: ParserWarning: Skipping line 22: expected 6 fields, saw 9
Skipping line 24: expected 6 fields, saw 19
Skipping line 40: expected 6 fields, saw 19
Skipping line 56: expected 6 fields, saw 9
Skipping line 95: expected 6 fields, saw 9

  pandas_dataset = pd.read_csv(


**Investigate what went wrong in the erroneous lines and maybe adjust the crawling process.** Your peers need to be able to read in your test dataset without errors.

In [4]:
from datasets import Dataset

# You could also load the Huggingface dataset directly from the tsv file but only if it does not contain erroneous lines
# Check the documentation on Huggingface for more options on loading data.
dataset = Dataset.from_pandas(pandas_dataset)
reduced_dataset = dataset.select_columns(["URL", "Text"])
# print(reduced_dataset[0:3])

In [5]:
# Let's remove empty articles
filtered_dataset = reduced_dataset.filter(lambda instance: not instance["Text"] == None)
# print(filtered_dataset[0:3])

Filter:   0%|          | 0/95 [00:00<?, ? examples/s]

In [6]:
# Let's keep track of the articles we removed. We might want to investigate what went wrong during crawling.
removed = reduced_dataset.filter(lambda instance: instance["Text"] == None)

print("URLs with empty articles:")
print(removed["URL"])

print("\nThis should add up:")
print(len(filtered_dataset), len(removed), len(reduced_dataset))

Filter:   0%|          | 0/95 [00:00<?, ? examples/s]

URLs with empty articles:
['https://montreal.ctvnews.ca/montreal-vegan-festival-shows-growing-popularity-of-plant-based-lifestyles-1.7104673', 'https://euroweeklynews.com/2024/11/07/vegan-option-for-the-mallorcan-classic-la-ensaimada/', 'https://www.bucksherald.co.uk/lifestyle/food-and-drink/greggs-christmas-menu-2024-4851499', 'https://www.dewsburyreporter.co.uk/lifestyle/food-and-drink/greggs-christmas-menu-2024-4851499', 'https://www.wakefieldexpress.co.uk/lifestyle/food-and-drink/greggs-christmas-menu-2024-4851499', 'https://www.northamptonchron.co.uk/lifestyle/food-and-drink/greggs-christmas-menu-2024-4851499', 'https://www.thescarboroughnews.co.uk/lifestyle/food-and-drink/greggs-christmas-menu-2024-4851499', 'https://www.derbyshiretimes.co.uk/lifestyle/food-and-drink/greggs-christmas-menu-2024-4851499', 'https://www.yorkshireeveningpost.co.uk/lifestyle/food-and-drink/greggs-christmas-menu-2024-4851499', 'https://www.bedfordtoday.co.uk/lifestyle/food-and-drink/greggs-christmas-men

## 2. Split into train and test

For NLP experiment, we want to split our dataset into training and test data. When working with machine learning models, the training data is further split into a training and a development portion. The development data is used for exploring hyperparameters and for finetuning the model. During the development phase, the test data is not touched at all. For many shared tasks, the test data is not even publicly available to avoid overfitting. 

In [7]:
seed = 5
splitted = filtered_dataset.train_test_split(test_size=0.2,seed=seed)
train = splitted["train"]
test = splitted["test"]
print(len(train), len(test), len(filtered_dataset))

44 12 56


Take a look at the source code of the method. **Why do we need to set a seed?**